# Fine-tuning SmolVLM with TRL

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch

In [ ]:
data_path = "drive/MyDrive/Amazon ML 2024/Dataset"
model_path = "drive/MyDrive/Amazon ML 2024/Models"

In [ ]:
train = pd.read_csv(os.path.join(data_path, "fine_tune_1.csv"), nrows=500)
train["entity_value"] = train["entity_value"].replace(np.nan, " unknown")
train["entity_name"] = train["entity_name"].str.replace("_", " ")

In [ ]:
test = pd.read_csv(os.path.join(data_path, "test.csv"), nrows=200)
test["entity_value"] = test["entity_value"].replace(np.nan, " unknown")
test["entity_name"] = test["entity_name"].str.replace("_", " ")

# 1. Install Dependencies

In [ ]:
pip install bitsandbytes peft trl

# 2. Load Dataset

In [ ]:
system_message = """You are a Vision Language Model specialized in interpreting visual data from product images.
Your task is to analyze product image and respond to queries with concise answers, usually with a number followed by unit.
The images include various product information like weight, length, width, etc.
Focus on delivering accurate, succinct answers based on the visual information. Avoid additional explanation unless absolutely necessary."""

In [ ]:
def format_data(image, entity_name, entity_value):
    return [
        # {
        #     "role": "system",
        #     "content": [
        #         {
        #             "type": "text",
        #             "text": system_message
        #         }
        #     ],
        # },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": f"What is the {entity_name} of this product?",
                }
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": entity_value
                }
            ],
        },
    ]

In [ ]:
from torch.utils.data import Dataset

class AmazonDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")

        return {
            "image": image,
            "entity_name": row["entity_name"],
            "entity_value": row["entity_value"]
        }

In [ ]:
train_dataset = AmazonDataset(train)

In [ ]:
test_dataset = AmazonDataset(test)

# 3. Load Model and Check Performance!

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"

In [ ]:
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

In [ ]:
from transformers import BitsAndBytesConfig

# Define the quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
# Load the model using the config
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.bfloat16,
    quantization_config=quantization_config,
    trust_remote_code=True,
    _attn_implementation="eager",
)

In [ ]:
print(model)

To evaluate the model's performance, we’ll use a sample from the dataset. First, let’s inspect the internal structure of this sample to understand how the data is organized.

In [ ]:
example = train_dataset[100]
example

We’ll use the sample without the system message to assess the VLM's raw understanding. Here’s the input we will use:

In [ ]:
format_data(**example)[0]

Now, let’s take a look at the product image corresponding to the sample. Can you answer the query based on the visual information?


In [ ]:
img = example['image']
plt.imshow(img)
plt.show()

Let’s create a method that takes the model, processor, and sample as inputs to generate the model's answer. This will allow us to streamline the inference process and easily evaluate the VLM's performance.

In [ ]:
def generate_text_from_sample(model, processor, sample, max_new_tokens=32, device="cuda"):
    formatted_sample = format_data(**sample)

    message = formatted_sample[:1]
    text_input = processor.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=False
    )

    image_inputs = []
    # Access the image from the formatted sample (user turn)
    image = formatted_sample[0]['content'][0]['image']
    if image.mode != 'RGB':
        image = image.convert('RGB')
    image_inputs.append(image)

    # Prepare the inputs for the model
    model_inputs = processor(
        text=text_input,
        images=image_inputs,
        return_tensors="pt",
    ).to("cuda", torch.float32)  # cast inputs to float32

    # Cast model inputs to bfloat16
    for k, v in model_inputs.items():
        if isinstance(v, torch.Tensor) and v.dtype == torch.float32:
            model_inputs[k] = v.to(torch.bfloat16)


    # Generate text with the model
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    # Trim the generated ids to remove the input ids
    trimmed_generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Decode the output text
    output_text = processor.batch_decode(
        trimmed_generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return output_text[0]  # Return the first decoded output text

In [ ]:
output = generate_text_from_sample(model, processor, example)
output

# 4. Fine-Tune the Model using TRL

In [ ]:
from peft import LoraConfig, get_peft_model

# Configure LoRA
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
    use_dora=True,
    init_lora_weights="gaussian"
)

In [ ]:
# Apply PEFT model adaptation
peft_model = get_peft_model(model, peft_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

In [ ]:
# image_token_id = processor.tokenizer.additional_special_tokens_ids[
#             processor.tokenizer.additional_special_tokens.index("<image>")]

def collate_fn(batch):
    texts, images = [], []
    for sample in batch:
        messages = format_data(sample["image"], sample["entity_name"], sample["entity_value"])
        text = processor.apply_chat_template(messages, add_generation_prompt=False)
        texts.append(text.strip())
        images.append(sample["image"])

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    for i in range(len(labels)):
          idx, prev = 0, -1
          for curr in batch["input_ids"][i]:
              if curr == 9531 and prev == 9519:
                  break
              prev = curr
              idx += 1
          labels[i][:idx+2] = -100
          labels[i][labels[i] == processor.tokenizer.pad_token_id] = -100

    batch["labels"] = labels
    return batch

In [ ]:
from trl import SFTConfig

output_dir = model_path + '/smolvlm_2'

# Configure training arguments using SFTConfig
training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=7,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    optim="adamw_torch_fused",
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    processing_class=processor,
)

Time to Train the Model!

In [ ]:
trainer.train()

Let's save the results

In [ ]:
trainer.save_model(training_args.output_dir)

# 5. Testing the Fine-Tuned Model

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"

In [ ]:
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

In [ ]:
from transformers import BitsAndBytesConfig

# Define the quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
# Load the model using the config
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
)

In [ ]:
adapter_path = model_path + '/smolvlm_2'
model.load_adapter(adapter_path)

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(model, adapter_path)

Let's evaluate the model on an unseen sample.


In [ ]:
format_data(**test_dataset[40])

In [ ]:
img = test_dataset[40]['image']
plt.imshow(img)
plt.show()

In [ ]:
output = generate_text_from_sample(model, processor, test_dataset[40])
output

In [ ]:
output = generate_text_from_sample(model, processor, example)
output